# WESAD — 05 · Designed-Experiment Statistics & Simpson's Paradox

*Project: When Wearable Stress Models Fail — Subject Generalization and Distribution Shift.*

Notebook 03 showed that honest (LOSO) accuracy is much lower than a leaked random-split number, and that the
gap is driven by **subject identity**. This notebook explains *why*, with the classical statistics the JD calls
out — analysis of variance for designed experiments, multiple comparisons, multicollinearity, confounding, and
Simpson's paradox — treating WESAD as what it is: a **within-subject repeated-measures experiment**.

Design decisions (the rigorous choices):
- **Aggregate-first.** Windows within a subject×condition are autocorrelated, so testing them as independent is
  pseudoreplication (fake power). The primary test uses one mean per subject×condition — a balanced 15×3 design.
- **Repeated-measures ANOVA** with subject as the within factor, **Greenhouse-Geisser** corrected for sphericity.
- **FDR across features**, because we test ~8 of them.
- **Simpson's paradox** is the mechanism linking these stats to the modelling: pooled relationships are
  confounded by subject and can *reverse* within-subject — which is exactly why subject-normalization and LOSO
  are mandatory.

> Input: `outputs/tables/wrist_features_w60_s60.csv`. numpy / scipy / sklearn only (no statsmodels needed —
> the RM-ANOVA, FDR and VIF are implemented here and were validated to match statsmodels exactly).

## 1 · Load & curate an interpretable feature set

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression, LogisticRegression

TAB = Path("outputs/tables"); FIG = Path("outputs/figures"); FIG.mkdir(parents=True, exist_ok=True)
feat = pd.read_csv(TAB/"wrist_features_w60_s60.csv")
names = {1:"baseline", 2:"stress", 3:"amusement"}
feat["cond"] = feat["condition"].map(names)

# ~8 interpretable features spanning modalities -> keeps the multiple-comparison story legible
FEATURES = [f for f in ["eda_mean","eda_std","eda_scr_count","eda_slope",
                        "temp_mean","temp_slope","hr_mean","acc_std"] if f in feat.columns]
CONDS = ["baseline","stress","amusement"]
print(f"{len(feat)} windows | features tested: {FEATURES}")
print("windows per subject x condition (should be balanced-ish):")
print(feat.pivot_table(index="subject", columns="cond", values="eda_mean", aggfunc="count").reindex(columns=CONDS))

## 2 · Dependency-free statistical primitives
Repeated-measures ANOVA (with Greenhouse-Geisser), Benjamini-Hochberg FDR, Holm post-hoc, and VIF.
Each was validated to match statsmodels before shipping.

In [ ]:
def rm_anova_oneway(wide):
    """One-way repeated-measures ANOVA. wide: (n_subjects, k_conditions), balanced."""
    wide = np.asarray(wide, float); n, k = wide.shape
    grand = wide.mean(); cond_means = wide.mean(0); subj_means = wide.mean(1)
    ss_cond = n*((cond_means-grand)**2).sum()
    ss_subj = k*((subj_means-grand)**2).sum()
    ss_err  = ((wide-grand)**2).sum() - ss_cond - ss_subj
    df_cond, df_err = k-1, (k-1)*(n-1)
    F = (ss_cond/df_cond)/(ss_err/df_err); p = stats.f.sf(F, df_cond, df_err)
    peta2 = ss_cond/(ss_cond+ss_err)
    S = np.cov(wide, rowvar=False); dbar=np.mean(np.diag(S)); gbar=S.mean(); rowbar=S.mean(1)
    eps = np.clip(k**2*(dbar-gbar)**2 / ((k-1)*((S**2).sum()-2*k*(rowbar**2).sum()+k**2*gbar**2)), 1/(k-1), 1.0)
    return dict(F=F, p=p, partial_eta2=peta2, gg_epsilon=eps, p_gg=stats.f.sf(F, df_cond*eps, df_err*eps))

def bh_fdr(pvals):
    p=np.asarray(pvals); m=len(p); order=np.argsort(p)
    crit=np.minimum.accumulate((p[order]*m/np.arange(1,m+1))[::-1])[::-1]
    out=np.empty(m); out[order]=np.minimum(crit,1.0); return out

def holm(pvals):
    p=np.asarray(pvals); m=len(p); order=np.argsort(p); adj=np.empty(m); run=0
    for rank,idx in enumerate(order):
        run=max(run,(m-rank)*p[idx]); adj[idx]=min(run,1.0)
    return adj

def vif(X):
    X=np.asarray(X,float); out=[]
    for j in range(X.shape[1]):
        Z=np.delete(X,j,axis=1); r2=LinearRegression().fit(Z,X[:,j]).score(Z,X[:,j])
        out.append(1/(1-r2) if r2<1 else np.inf)
    return np.array(out)
print("primitives defined")

## 3 · Repeated-measures ANOVA + FDR

For each feature: aggregate to one mean per subject×condition, run RM-ANOVA (GG-corrected), collect the
p-value and effect size, then FDR-correct across features. The result to hunt for is a feature significant
*uncorrected* but not after FDR — the concrete "a metric moved but it isn't real" artifact.

In [ ]:
agg = feat.groupby(["subject","cond"])[FEATURES].mean().reset_index()

rows=[]
for f in FEATURES:
    wide = agg.pivot(index="subject", columns="cond", values=f)[CONDS].dropna()
    r = rm_anova_oneway(wide.values)
    rows.append(dict(feature=f, F=r["F"], p_raw=r["p"], p_gg=r["p_gg"],
                     partial_eta2=r["partial_eta2"], gg_epsilon=r["gg_epsilon"], n_subj=len(wide)))
res = pd.DataFrame(rows)
res["p_fdr"] = bh_fdr(res["p_gg"].values)
res["sig_raw"] = res["p_gg"] < 0.05
res["sig_fdr"] = res["p_fdr"] < 0.05
res = res.sort_values("p_fdr").reset_index(drop=True)
res.to_csv(TAB/"rm_anova_fdr.csv", index=False)
pd.set_option("display.float_format", lambda x: f"{x:.4g}")
print(res[["feature","F","partial_eta2","gg_epsilon","p_gg","p_fdr","sig_raw","sig_fdr"]].to_string(index=False))
lost = res[(res.sig_raw) & (~res.sig_fdr)]
print("\nSignificant uncorrected but NOT after FDR:", list(lost.feature) or "none",
      "\n(effect sizes matter as much as p: partial_eta2 is the practical magnitude)")

### 3.1 · Post-hoc pairwise contrasts (Holm-corrected)
For FDR-significant features, which conditions actually differ? Paired t-tests across subjects with Holm
correction over the three contrasts. Watch for features that separate stress from baseline but **not** stress
from amusement — that's the arousal-vs-valence distinction (both stress and amusement are high-arousal).

In [ ]:
pairs = [("baseline","stress"),("stress","amusement"),("baseline","amusement")]
ph=[]
for f in res[res.sig_fdr].feature:
    w = agg.pivot(index="subject", columns="cond", values=f)[CONDS].dropna()
    praw=[stats.ttest_rel(w[a], w[b]).pvalue for a,b in pairs]
    padj=holm(praw)
    for (a,b),pr,pa in zip(pairs,praw,padj):
        ph.append(dict(feature=f, contrast=f"{a} vs {b}",
                       mean_diff=w[a].mean()-w[b].mean(), p_holm=pa, sig=pa<0.05))
ph=pd.DataFrame(ph)
if len(ph):
    ph.to_csv(TAB/"posthoc_holm.csv", index=False)
    print(ph.to_string(index=False))
else:
    print("no FDR-significant features to follow up")

## 4 · Multicollinearity (VIF)

The physiological features are driven by the same sympathetic response, so they're correlated — which makes
individual regression coefficients unstable and uninterpretable, and is why we rely on regularized models
rather than reading coefficients. High VIF here is the formal justification.

In [ ]:
Z = feat[FEATURES].dropna().values
Zs = (Z - Z.mean(0))/Z.std(0)
v = pd.DataFrame(dict(feature=FEATURES, VIF=vif(Zs))).sort_values("VIF", ascending=False)
v.to_csv(TAB/"vif.csv", index=False)
print(v.to_string(index=False))
print("\nRule of thumb: VIF>5 notable, >10 severe. Collinear features -> don't interpret single coefficients.")

## 5 · Simpson's paradox — the mechanism behind the leakage

The link between the statistics and the modelling. Because subjects have very different baselines (ICC 0.85–0.95
from notebook 01), a feature relationship measured on **pooled** data can **reverse** once you look **within**
each subject. That reversal is why pooled models are confounded and why subject-normalization / LOSO are
mandatory.

**Search:** for every feature pair, compare the pooled correlation sign to the mean within-subject correlation
sign. Flag reversals.

In [ ]:
from itertools import combinations
flips=[]
for fa, fb in combinations(FEATURES, 2):
    sub = feat[["subject",fa,fb]].dropna()
    pooled = np.corrcoef(sub[fa], sub[fb])[0,1]
    wr=[]
    for s,g in sub.groupby("subject"):
        if len(g)>=5 and g[fa].std()>0 and g[fb].std()>0:
            wr.append(np.corrcoef(g[fa], g[fb])[0,1])
    within = np.mean(wr)
    flips.append(dict(fa=fa, fb=fb, pooled_r=pooled, within_r=within,
                      sign_flip=(np.sign(pooled)!=np.sign(within)), gap=abs(pooled-within)))
flips=pd.DataFrame(flips).sort_values("gap", ascending=False)
flips.to_csv(TAB/"simpson_pairs.csv", index=False)
print("Feature pairs where pooled and within-subject correlations DISAGREE in sign:")
show = flips[flips.sign_flip]
print((show if len(show) else flips.head()).to_string(index=False))

### 5.1 · Visualise the strongest reversal
Per-subject points (coloured by subject) with per-subject regression lines, and the pooled line cutting across
them. When the pooled line slopes opposite to the within-subject lines, that is Simpson's paradox — and the
visual justification for LOSO.

In [ ]:
cand = flips[flips.sign_flip]
pair = (cand.iloc[0] if len(cand) else flips.iloc[0])
fa, fb = pair["fa"], pair["fb"]
sub = feat[["subject",fa,fb]].dropna()

plt.figure(figsize=(7,6))
cmap = plt.cm.tab20(np.linspace(0,1,sub.subject.nunique()))
for (s,g),c in zip(sub.groupby("subject"), cmap):
    plt.scatter(g[fa], g[fb], s=14, color=c, alpha=0.6)
    if len(g)>=3 and g[fa].std()>0:
        b=np.polyfit(g[fa],g[fb],1); xs=np.array([g[fa].min(),g[fa].max()]); plt.plot(xs, np.polyval(b,xs), color=c, lw=1)
bp=np.polyfit(sub[fa],sub[fb],1); xs=np.array([sub[fa].min(),sub[fa].max()])
plt.plot(xs, np.polyval(bp,xs), "k--", lw=3, label=f"POOLED (r={pair['pooled_r']:.2f})")
plt.xlabel(fa); plt.ylabel(fb)
plt.title(f"Simpson's paradox: pooled r={pair['pooled_r']:.2f} vs within-subject r={pair['within_r']:.2f}\n"
          f"(each colour = one subject)")
plt.legend(); plt.tight_layout(); plt.savefig(FIG/"simpson_paradox.png", dpi=140); plt.show()

### 5.2 · Simpson's in the modelling language
Same reversal, expressed as a regression coefficient sign. Predict *stress vs. not* from one feature, pooled
vs. with subject fixed effects. A sign change confirms that pooling confounds the feature effect with subject
identity — precisely what subject-normalization removes and what inflated the random-split score in nb 03.

In [ ]:
f0 = FEATURES[0] if flips.iloc[0]["fa"]=="__" else flips[flips.sign_flip].iloc[0]["fa"] if flips.sign_flip.any() else "temp_mean"
f0 = f0 if f0 in feat.columns else "eda_mean"
sub = feat[["subject","condition",f0]].dropna().copy()
sub["is_stress"] = (sub["condition"]==2).astype(int)
xz = (sub[[f0]].values - sub[[f0]].values.mean())/sub[[f0]].values.std()

pooled = LogisticRegression(max_iter=2000).fit(xz, sub.is_stress)
D = pd.get_dummies(sub["subject"], drop_first=True).values.astype(float)
withinX = np.hstack([xz, D])
withinm = LogisticRegression(max_iter=5000).fit(withinX, sub.is_stress)
print(f"Feature: {f0}  (predicting stress)")
print(f"  pooled logistic coef            : {pooled.coef_[0,0]:+.3f}")
print(f"  subject-fixed-effects coef      : {withinm.coef_[0,0]:+.3f}")
print("  => sign flip" if np.sign(pooled.coef_[0,0])!=np.sign(withinm.coef_[0,0])
      else "  => same sign (magnitude/So still shifts with subject adjustment)")

## 6 · Findings & handoff

Fill from your run:
- **Condition effects (RM-ANOVA + FDR):** `[__]` of `[__]` features survive FDR; effect sizes (partial η²)
  `[__]`. `[feature]` was significant uncorrected but not after FDR — the "moved but not real" example.
- **Post-hoc:** stress separates from baseline on `[features]`; stress vs amusement is `[harder/easier]`
  (arousal-vs-valence).
- **Multicollinearity:** max VIF `[__]` (`[features]`) — justifies regularization, forbids coefficient-level claims.
- **Simpson's paradox:** `[feature pair]` reverses sign (pooled r=`[__]`, within r=`[__]`); the stress-coef
  `[flips/shrinks]` under subject fixed effects. This *is* the mechanism behind the subject-leakage gap in nb 03.

**Remaining work:**
- **`04_slurm_sweep`** — LOSO × seed × config as a SLURM array: overlap-leakage (shift 0.25 s), and the
  raw-vs-subject-normalized experiment (expected to shrink the LOSO gap — the applied payoff of this notebook).
- **Package** all notebooks + `outputs/` into the project repo structure (src/, configs, README, model card).

Upload this notebook's `outputs/tables/*.csv` and `outputs/figures/simpson_paradox.png`.